# AML GNN — Data Preparation v3: GFP Structural Features + FX-Corrected Amounts

Drop-in replacement for `Data_prepration.ipynb`.
Keeps the **original baseline edge/node features** from `Data_prepration.ipynb` (Section 1)
and appends **Graph Feature Preprocessor (GFP) structural features** from IBM SnapML (Section 2).

The only change to Section 1 is that `Amount_Log` now uses **day-specific FX rates** fetched
from Yahoo Finance so that amounts are truly comparable across the 7 currencies in LI-Small.

## Feature inventory (v3)

| Section | Group | Cols | Key AML signal |
|---|---|---|---|
| 1 | Transaction-level baseline (FX-corrected) | 16 | Amount in USD, OHE payment format, currency mismatch, timing, entity type |
| 2 | GFP: Fan in/out histogram | 4 | Fan-in/out degree buckets over time window |
| 2 | GFP: Degree histogram | 4 | In/out degree buckets |
| 2 | GFP: Scatter-Gather histogram | 2 | Scatter-gather pattern buckets |
| 2 | GFP: Temporal-cycle histogram | 2 | Cycle-closing edges in time window |
| 2 | GFP: Vertex stats (source, out) | ~8 | Fan, degree, ratio, avg, sum, var, skew, kurtosis of Amount USD |
| 2 | GFP: Vertex stats (source, in) | ~8 | Same stats for incoming edges |
| 2 | GFP: Vertex stats (dest, out) | ~8 | Same stats for destination outgoing |
| 2 | GFP: Vertex stats (dest, in) | ~8 | Same stats for destination incoming |

## Design principles

- **No data leakage:** GFP is fit on the full sorted transaction stream — it uses only
  transactions with timestamp < t for every feature of edge (u→v, t), by design of the library.
- **Original node features preserved:** `Bank_ID_Norm`, `EntityType` OHE (5 cols) — same as baseline.
- **Output files:** `edge_features_gfp.csv`, `node_features_gfp.csv`,
  `train/val/test_graph_gfp.pt`, `account_to_idx_gfp.pkl`.
- **EDGE_DIM** is printed at the end — update your model notebook accordingly.


---
## 0. Imports & Setup


In [1]:
import warnings, time, json, pickle
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import yfinance as yf

import torch
from torch_geometric.data import Data
from tqdm import tqdm

# SnapML Graph Feature Preprocessor
from snapml import GraphFeaturePreprocessor

class Timer:
    def __init__(self, label): self.label = label
    def __enter__(self): self.t = time.time(); return self
    def __exit__(self, *a): print(f'  [{self.label}] done in {time.time()-self.t:.1f}s')

print('Libraries loaded ✓')

Libraries loaded ✓


In [2]:
df_tr = pd.read_csv('Data/LI-Small_Trans.csv', low_memory=False)
df_ac = pd.read_csv('Data/LI-Small_accounts.csv', low_memory=False)

print(f'Transactions : {len(df_tr):,}')
print(f'Accounts     : {len(df_ac):,}')
df_tr.head(3)

Transactions : 6,924,049
Accounts     : 712,688


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:08,11,8000ECA90,11,8000ECA90,3195403.00,US Dollar,3195403.00,US Dollar,Reinvestment,0
1,2022/09/01 00:21,3402,80021DAD0,3402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,Reinvestment,0
2,2022/09/01 00:00,11,8000ECA90,1120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,0


---
## 1. Transaction-level baseline features  *(v3: FX-corrected Amount_Log)*

Identical to the original `Data_prepration.ipynb` **except** that `Amount_Log` is computed
on the USD-converted amount rather than the raw `Amount Paid`.

**Why FX-correct the amount?**
LI-Small contains transactions in multiple currencies (USD, EUR, GBP, BTC, …).  
A raw `Amount Paid = 1000` means $1,000 if the currency is USD but ~$63M if it is Bitcoin.
Without conversion, the model sees structurally identical laundering patterns at wildly
different scales depending on the currency, making it impossible to learn a unified
amount-based signal.  Converting to USD first makes the distribution coherent.

**FX approach:** daily close rates fetched from Yahoo Finance for the exact date range
of the dataset (Sep 2022).  Weekends are forward-filled from the last trading close.
USD rows keep rate = 1.0 by definition.

All other features (OHE payment format, currency mismatch, time cyclicals, entity type, bank)
are unchanged from the baseline.


In [3]:
# ── 1.1  Fetch daily FX rates from Yahoo Finance ─────────────────────────────
# Map payment currency names to Yahoo Finance tickers.
# Crypto pairs use {ISO}-USD; FX pairs use {ISO}USD=X.
CURRENCY_TO_ISO = {
    'US Dollar':        'USD',
    'Euro':             'EUR',
    'British Pound':    'GBP',
    'Australian Dollar':'AUD',
    'Canadian Dollar':  'CAD',
    'Swiss Franc':      'CHF',
    'Chinese Yuan':     'CNY',
    'Japanese Yen':     'JPY',
    'Mexican Peso':     'MXN',
    'Brazilian Real':   'BRL',
    'Indian Rupee':     'INR',
    'South Korean Won': 'KRW',
    'Russian Ruble':    'RUB',
    'Saudi Riyal':      'SAR',
    'Singapore Dollar': 'SGD',
    'Hong Kong Dollar': 'HKD',
    'Norwegian Krone':  'NOK',
    'Swedish Krona':    'SEK',
    'Danish Krone':     'DKK',
    'New Zealand Dollar':'NZD',
    'South African Rand':'ZAR',
    'Turkish Lira':     'TRY',
    'UAE Dirham':       'AED',
    'Bitcoin':          'BTC',
    'Ethereum':         'ETH',
}

_all_currencies = df_tr['Payment Currency'].unique()
_non_usd        = [c for c in _all_currencies if c != 'US Dollar' and c in CURRENCY_TO_ISO]
print(f'Non-USD currencies in dataset: {_non_usd}')

# Date range: Sep 2022 with buffer.  Forward-fill covers weekends.
_START      = '2022-09-01'
_END        = '2022-09-19'
_full_dates = pd.date_range(start=_START, end='2022-09-17')

fx_rate_lookup = {}   # {(currency_name, 'YYYY-MM-DD'): rate_to_USD}

for _cname in _non_usd:
    _iso    = CURRENCY_TO_ISO[_cname]
    _ticker = f'{_iso}-USD' if _iso in ('BTC', 'ETH') else f'{_iso}USD=X'
    try:
        _raw = yf.download(_ticker, start=_START, end=_END,
                           auto_adjust=True, progress=False)
        if _raw.empty:
            print(f'  WARNING: no data for {_ticker}, defaulting to 1.0')
            continue
        _cl = _raw['Close']
        if hasattr(_cl, 'columns'):   # handle MultiIndex from some yfinance versions
            _cl = _cl.iloc[:, 0]
        _cl = _cl.squeeze()
        _cl = _cl.reindex(_full_dates).ffill().bfill()   # fill weekends
        for _dt, _rate in _cl.items():
            if pd.notna(_rate):
                fx_rate_lookup[(_cname, str(_dt.date()))] = float(_rate)
        print(f'  {_cname:25s} ({_ticker}): {_cl.notna().sum()} days, '
              f'last={float(_cl.iloc[-1]):.4f}')
    except Exception as _e:
        print(f'  ERROR fetching {_ticker}: {_e}')

print(f'\nFX lookup entries: {len(fx_rate_lookup):,}  (USD rows get rate=1.0 inline)')
print('FX rates loaded ✓')

Non-USD currencies in dataset: ['Euro', 'Bitcoin', 'Australian Dollar', 'Canadian Dollar', 'Mexican Peso', 'Swiss Franc', 'Saudi Riyal']
  Euro                      (EURUSD=X): 17 days, last=0.9988
  Bitcoin                   (BTC-USD): 17 days, last=20127.5762
  Australian Dollar         (AUDUSD=X): 17 days, last=0.6687
  Canadian Dollar           (CADUSD=X): 17 days, last=0.7549
  Mexican Peso              (MXNUSD=X): 17 days, last=0.0498
  Swiss Franc               (CHFUSD=X): 17 days, last=1.0397
  Saudi Riyal               (SARUSD=X): 17 days, last=0.2665

FX lookup entries: 119  (USD rows get rate=1.0 inline)
FX rates loaded ✓


In [4]:
# ── 1.2  Build edge dataframe with all baseline features ─────────────────────
edges = df_tr.copy()
edges['Timestamp'] = pd.to_datetime(edges['Timestamp'])
edges = edges.sort_values('Timestamp', kind='stable').reset_index(drop=True)

# Training cutoff (60%) — used for leakage-free target encodings later
_t1_idx = int(len(edges) * 0.60)

# ── Amount in USD + log1p ──────────────────────────────────────────────────
# Vectorised: build (currency, date) → rate via unique pairs, then left-merge
edges['_date_str'] = edges['Timestamp'].dt.date.astype(str)
_pairs = edges[['Payment Currency', '_date_str']].drop_duplicates().copy()
_pairs['_fx_rate'] = _pairs.apply(
    lambda r: fx_rate_lookup.get((r['Payment Currency'], r['_date_str']), 1.0),
    axis=1,
)
edges = edges.merge(_pairs, on=['Payment Currency', '_date_str'], how='left')
edges['_fx_rate']  = edges['_fx_rate'].fillna(1.0)
edges['Amount_USD'] = edges['Amount Paid'] * edges['_fx_rate']   # raw USD amount (used by GFP)
edges['Amount_Log'] = np.log1p(edges['Amount_USD'])               # log1p-compressed for GNN

# ── Currency mismatch ─────────────────────────────────────────────────────
edges['Currency_Mismatch'] = (
    edges['Receiving Currency'] != edges['Payment Currency']
).astype(np.int8)

# ── Cyclical time encoding ────────────────────────────────────────────────
_hour = edges['Timestamp'].dt.hour
_dow  = edges['Timestamp'].dt.dayofweek
edges['Hour_Sin']      = np.sin(2 * np.pi * _hour / 24)
edges['Hour_Cos']      = np.cos(2 * np.pi * _hour / 24)
edges['DayOfWeek_Sin'] = np.sin(2 * np.pi * _dow  / 7)
edges['DayOfWeek_Cos'] = np.cos(2 * np.pi * _dow  / 7)
edges['Is_Weekend']    = (_dow >= 5).astype(np.int8)

# ── Payment Format OHE  (7 dummies — same as baseline) ───────────────────
_fmt_dummies = pd.get_dummies(edges['Payment Format'], prefix='PayFmt').astype(np.int8)
edges = pd.concat([edges, _fmt_dummies], axis=1)
PAY_FMT_COLS = list(_fmt_dummies.columns)

# ── Is ACH flag ───────────────────────────────────────────────────────────
edges['Is_ACH'] = (edges['Payment Format'] == 'ACH').astype(np.int8)

# ── Self-loop ─────────────────────────────────────────────────────────────
edges['Is_Self_Loop'] = (edges['Account'] == edges['Account.1']).astype(np.int8)

# ── Rename for clarity ────────────────────────────────────────────────────
edges = edges.rename(columns={
    'Account':       'src_account',
    'Account.1':     'dst_account',
    'From Bank':     'src_bank',
    'To Bank':       'dst_bank',
    'Is Laundering': 'label',
})

# Baseline feature columns (same as original Data_prepration.ipynb
# except Amount_Log is now FX-corrected)
BASE_EDGE_COLS = (
    ['Amount_Log', 'Currency_Mismatch',
     'Hour_Sin', 'Hour_Cos', 'DayOfWeek_Sin', 'DayOfWeek_Cos',
     'Is_Weekend', 'Is_ACH', 'Is_Self_Loop']
    + PAY_FMT_COLS
)

print(f'Baseline edge features : {len(BASE_EDGE_COLS)}')
print(f'  {BASE_EDGE_COLS}')
print(f'\nAmount_USD stats:')
print(edges['Amount_USD'].describe().round(2))

Baseline edge features : 16
  ['Amount_Log', 'Currency_Mismatch', 'Hour_Sin', 'Hour_Cos', 'DayOfWeek_Sin', 'DayOfWeek_Cos', 'Is_Weekend', 'Is_ACH', 'Is_Self_Loop', 'PayFmt_ACH', 'PayFmt_Bitcoin', 'PayFmt_Cash', 'PayFmt_Cheque', 'PayFmt_Credit Card', 'PayFmt_Reinvestment', 'PayFmt_Wire']

Amount_USD stats:
count    6.924049e+06
mean     4.563286e+06
std      1.541978e+09
min      0.000000e+00
25%      2.136600e+02
50%      1.482220e+03
75%      1.199899e+04
max      3.644854e+12
Name: Amount_USD, dtype: float64


---
## 2. Node features  *(unchanged from baseline)*

- `Bank_ID_Norm`  — min-max normalised integer bank ID
- `EntityType` OHE — Corporation / Individual / Partnership / Sole Proprietor / Other (5 cols)

Keeping the original node features avoids the target-encoding instability identified
as the main cause of the v2 AUC regression (only 1,813 training positives → noisy rates).


In [5]:
nodes = df_ac.copy()
nodes = nodes.drop_duplicates(subset='Account Number', keep='first').reset_index(drop=True)

# ── Entity type ───────────────────────────────────────────────────────────
def _extract_entity_type(name):
    for t in ('Corporation', 'Individual', 'Partnership', 'Sole'):
        if t.lower() in str(name).lower():
            return t
    return 'Other'

nodes['Entity_Type'] = nodes['Entity Name'].apply(_extract_entity_type)

# ── Entity type OHE ───────────────────────────────────────────────────────
_et_dummies = pd.get_dummies(nodes['Entity_Type'], prefix='EntityType').astype(np.int8)
nodes = pd.concat([nodes, _et_dummies], axis=1)
ENTITY_TYPE_COLS = list(_et_dummies.columns)

# ── Bank ID normalised ────────────────────────────────────────────────────
_bmin, _bmax  = nodes['Bank ID'].min(), nodes['Bank ID'].max()
nodes['Bank_ID_Norm'] = (
    (nodes['Bank ID'] - _bmin) / (_bmax - _bmin + 1e-9)
).astype(np.float32)

NODE_FEAT_COLS = ['Bank_ID_Norm'] + ENTITY_TYPE_COLS
node_features  = nodes[['Account Number'] + NODE_FEAT_COLS].rename(
    columns={'Account Number': 'account_id'}
)

print(f'Node feature matrix: {node_features.shape}')
print(f'  Features: {NODE_FEAT_COLS}')
print(f'\nEntity type distribution:')
print(nodes['Entity_Type'].value_counts())

Node feature matrix: (712684, 6)
  Features: ['Bank_ID_Norm', 'EntityType_Corporation', 'EntityType_Individual', 'EntityType_Partnership', 'EntityType_Sole']

Entity type distribution:
Entity_Type
Corporation    268889
Partnership    225814
Sole           217130
Individual        851
Name: count, dtype: int64


---
## 3. Graph Feature Preprocessor (GFP) structural features

IBM SnapML's `GraphFeaturePreprocessor` computes the AML-specific graph patterns
described in Altman et al. Appendix D for every transaction in a single pass.

**Causal correctness:** GFP processes edges in timestamp order and uses only
edges with timestamp < t when computing features for edge (u→v, t).  
No leakage by construction.

### Patterns computed

| Pattern | Parameter | AML signal |
|---|---|---|
| Fan in/out | `fan`, bins=[2,3], window=24h | Fan-in mule accumulation, fan-out scatter |
| Degree in/out | `degree`, bins=[2,3], window=24h | Account activity intensity |
| Scatter-Gather | `scatter-gather`, bins=[2,3], window=6h | Rapid gather-then-scatter |
| Temporal cycle | `temp-cycle`, bins=[2,3], window=24h | Money loop closure |
| Vertex stats | source+dest, in+out, Amount_USD | Fan, degree, ratio, avg, sum, var, skew, kurtosis |

### Column naming convention

GFP returns a numpy array.  `build_gfp_colnames()` below reconstructs the
column names from the `params` dict — same logic as the reference notebook.


In [6]:
# ── 3.1  GFP parameters ───────────────────────────────────────────────────
GFP_PARAMS = {
    'num_threads':  4,
    'time_window':  24 * 60 * 60,      # 24-hour default window

    # Vertex statistics: source & dest × in & out, on Amount_USD (col index 3)
    'vertex_stats':       True,
    'vertex_stats_cols':  [3],          # column 3 = Amount_USD in the input frame
    # 0:fan, 1:deg, 2:ratio, 3:avg, 4:sum, 5:min, 6:max, 7:median, 8:var, 9:skew, 10:kurtosis
    'vertex_stats_feats': [0, 1, 2, 3, 4, 8, 9, 10],

    # Fan in/out
    'fan':       True,
    'fan_bins':  [2, 3],               # 3 bins: [2-3), [3-inf)

    # Degree in/out
    'degree':      True,
    'degree_bins': [2, 3],

    # Scatter-Gather (short 6h window)
    'scatter-gather':      True,
    'scatter-gather_tw':   6 * 60 * 60,
    'scatter-gather_bins': [2, 3],

    # Temporal cycle
    'temp-cycle':      True,
    'temp-cycle_bins': [2, 3],

    # Length-constrained simple cycle (expensive — disabled)
    'lc-cycle': False,
}

print('GFP parameters:')
print(json.dumps(GFP_PARAMS, indent=4))

GFP parameters:
{
    "num_threads": 4,
    "time_window": 86400,
    "vertex_stats": true,
    "vertex_stats_cols": [
        3
    ],
    "vertex_stats_feats": [
        0,
        1,
        2,
        3,
        4,
        8,
        9,
        10
    ],
    "fan": true,
    "fan_bins": [
        2,
        3
    ],
    "degree": true,
    "degree_bins": [
        2,
        3
    ],
    "scatter-gather": true,
    "scatter-gather_tw": 21600,
    "scatter-gather_bins": [
        2,
        3
    ],
    "temp-cycle": true,
    "temp-cycle_bins": [
        2,
        3
    ],
    "lc-cycle": false
}


In [7]:
def build_gfp_colnames(params):
    """
    Reconstruct the GFP output column names from the params dict.
    Mirrors the logic in the reference Graph_Feature_Preprocessor notebook.
    Returns a list of strings (one per GFP output column, excluding the
    first 5 input pass-through columns: txID, src, dst, ts, amount).
    """
    colnames = []

    # Pattern histogram columns
    for pattern in ['fan', 'degree', 'scatter-gather', 'temp-cycle', 'lc-cycle']:
        if params.get(pattern, False):
            bins = params[pattern + '_bins']
            n    = len(bins)
            if pattern in ['fan', 'degree']:
                for side in ['in', 'out']:
                    for i in range(n - 1):
                        colnames.append(f'{pattern}_{side}_bins_{bins[i]}-{bins[i+1]}')
                    colnames.append(f'{pattern}_{side}_bins_{bins[-1]}-inf')
            else:
                for i in range(n - 1):
                    colnames.append(f'{pattern}_bins_{bins[i]}-{bins[i+1]}')
                colnames.append(f'{pattern}_bins_{bins[-1]}-inf')

    # Vertex statistics columns
    vert_feat_names = ['fan', 'deg', 'ratio', 'avg', 'sum', 'min', 'max',
                       'median', 'var', 'skew', 'kurtosis']
    for orig in ['source', 'dest']:
        for direction in ['out', 'in']:
            for k in [0, 1, 2]:
                if k in params['vertex_stats_feats']:
                    colnames.append(f'{orig}_{vert_feat_names[k]}_{direction}')
            for col in params['vertex_stats_cols']:
                for k in [3, 4, 5, 6, 7, 8, 9, 10]:
                    if k in params['vertex_stats_feats']:
                        colnames.append(f'{orig}_{vert_feat_names[k]}_col{col}_{direction}')

    return colnames


GFP_FEAT_COLS = build_gfp_colnames(GFP_PARAMS)
print(f'GFP will produce {len(GFP_FEAT_COLS)} feature columns:')
for c in GFP_FEAT_COLS:
    print(f'  {c}')

GFP will produce 44 feature columns:
  fan_in_bins_2-3
  fan_in_bins_3-inf
  fan_out_bins_2-3
  fan_out_bins_3-inf
  degree_in_bins_2-3
  degree_in_bins_3-inf
  degree_out_bins_2-3
  degree_out_bins_3-inf
  scatter-gather_bins_2-3
  scatter-gather_bins_3-inf
  temp-cycle_bins_2-3
  temp-cycle_bins_3-inf
  source_fan_out
  source_deg_out
  source_ratio_out
  source_avg_col3_out
  source_sum_col3_out
  source_var_col3_out
  source_skew_col3_out
  source_kurtosis_col3_out
  source_fan_in
  source_deg_in
  source_ratio_in
  source_avg_col3_in
  source_sum_col3_in
  source_var_col3_in
  source_skew_col3_in
  source_kurtosis_col3_in
  dest_fan_out
  dest_deg_out
  dest_ratio_out
  dest_avg_col3_out
  dest_sum_col3_out
  dest_var_col3_out
  dest_skew_col3_out
  dest_kurtosis_col3_out
  dest_fan_in
  dest_deg_in
  dest_ratio_in
  dest_avg_col3_in
  dest_sum_col3_in
  dest_var_col3_in
  dest_skew_col3_in
  dest_kurtosis_col3_in


In [8]:
# ── 3.2  Build GFP input dataframe ───────────────────────────────────────
# GFP expects exactly 5 columns in this order:
#   [transactionID, sourceAccountID (int), targetAccountID (int), timestamp (int seconds), amount]
#
# Timestamp: GFP uses relative integer seconds.  We subtract the epoch start
# (first transaction timestamp) so the values start near 0 — avoids int overflow.

# Build integer account index (required by GFP — it does not accept string IDs)
_all_accs     = pd.concat([edges['src_account'], edges['dst_account']]).unique()
account_to_idx = {acc: idx for idx, acc in enumerate(_all_accs)}

_ts_epoch = int(edges['Timestamp'].astype('int64').min() // 10**9)  # seconds since Unix epoch
edges['_ts_sec'] = (edges['Timestamp'].astype('int64') // 10**9) - _ts_epoch
edges['_src_idx'] = edges['src_account'].map(account_to_idx)
edges['_dst_idx'] = edges['dst_account'].map(account_to_idx)

# GFP requires contiguous integer transaction IDs
edges['_txn_id'] = np.arange(len(edges))

gfp_input = edges[['_txn_id', '_src_idx', '_dst_idx', '_ts_sec', 'Amount_USD']].copy()
gfp_input.columns = ['transactionID', 'sourceAccountID', 'targetAccountID',
                     'timestamp', 'Amount_USD']

print(f'GFP input shape : {gfp_input.shape}')
print(f'Timestamp range : {gfp_input["timestamp"].min():,}  →  {gfp_input["timestamp"].max():,} s')
print(f'Accounts        : {len(account_to_idx):,}')
gfp_input.head(3)

GFP input shape : (6924049, 5)
Timestamp range : 0  →  1,438,080 s
Accounts        : 705,903


,transactionID,sourceAccountID,targetAccountID,timestamp,Amount_USD
0,0,0,316753,0,592571.00
1,1,1,1,0,2941.56
2,2,2,394190,0,0.36


In [9]:
# ── 3.3  Run GFP ─────────────────────────────────────────────────────────
# fit_transform processes all edges in timestamp order, using only
# past transactions when computing features for each edge — no leakage.
#
# Runtime: ~10-30 min on 6.9M edges depending on num_threads and hardware.
# The output is a 2D numpy array of shape (n_edges, 5 + n_gfp_features).

print('Initialising GraphFeaturePreprocessor ...')
gfp = GraphFeaturePreprocessor()
gfp.set_params(GFP_PARAMS)

print(f'Running GFP on {len(gfp_input):,} edges ...')
print(f'  num_threads = {GFP_PARAMS["num_threads"]}')
print(f'  Expected output cols: 5 (pass-through) + {len(GFP_FEAT_COLS)} (new) = {5 + len(GFP_FEAT_COLS)}')
print('  This may take 10–30 minutes ...')

with Timer('GFP fit_transform'):
    gfp_output = gfp.fit_transform(gfp_input)  # numpy array shape (n_edges, 5 + n_gfp)

print(f'GFP output shape: {gfp_output.shape}')
assert gfp_output.shape[0] == len(edges), 'Row count mismatch between GFP output and edges!'
assert gfp_output.shape[1] == 5 + len(GFP_FEAT_COLS), (
    f'Expected {5 + len(GFP_FEAT_COLS)} cols, got {gfp_output.shape[1]}'
)
print('Shape check ✓')

Initialising GraphFeaturePreprocessor ...


AttributeError: module 'snapml.libsnapmllocal3_avx2' has no attribute 'gf_allocate'

In [ ]:
# ── 3.4  Extract GFP features into a DataFrame ───────────────────────────
# gfp_output columns: [txID, src, dst, ts, amount, gfp_feat_0, gfp_feat_1, ...]
# We skip the first 5 pass-through columns and keep only the new features.

gfp_feats_df = pd.DataFrame(
    gfp_output[:, 5:].astype(np.float32),
    columns=GFP_FEAT_COLS,
)

# Sanity: verify txID alignment (col 0 should be 0, 1, 2, ...)
_txn_ids = gfp_output[:, 0].astype(int)
assert (_txn_ids == np.arange(len(edges))).all(), \
    'Transaction IDs in GFP output are not aligned with edges DataFrame!'
print('Transaction ID alignment verified ✓')

print(f'\nGFP feature stats (first 5 columns):')
print(gfp_feats_df.iloc[:, :5].describe().round(3))

---
## 4. Assemble final edge feature matrix & sanity checks


In [ ]:
# ── 4.1  Concatenate baseline + GFP features ─────────────────────────────
EDGE_FEAT_COLS = BASE_EDGE_COLS + GFP_FEAT_COLS

edge_features = pd.concat([
    edges[['src_account', 'dst_account', 'label', 'Timestamp'] + BASE_EDGE_COLS]
        .reset_index(drop=True),
    gfp_feats_df.reset_index(drop=True),
], axis=1)

# Fill any NaN (first-transaction rows where GFP can't compute history)
nan_count = edge_features[EDGE_FEAT_COLS].isna().sum().sum()
edge_features[EDGE_FEAT_COLS] = edge_features[EDGE_FEAT_COLS].fillna(0).astype(np.float32)

print('── Feature inventory ──────────────────────────────────────────────')
print(f'  Baseline features  : {len(BASE_EDGE_COLS):>3}  (FX-corrected Amount_Log + OHE + timing)')
print(f'  GFP features       : {len(GFP_FEAT_COLS):>3}  (fan, degree, sg, temp-cycle, vertex stats)')
print(f'  ─────────────────────────────')
print(f'  TOTAL edge_attr dim: {len(EDGE_FEAT_COLS):>3}')
print(f'\nNaN values filled to 0: {nan_count:,}')
print(f'Edge feature matrix : {edge_features.shape}')

In [ ]:
# ── 4.2  Sanity check: laundering vs legitimate mean feature ratios ───────
# Ratio > 1 means laundering transactions show higher values — expected for AML signals.
signal_cols = [
    'Amount_Log',
    'fan_in_bins_2-3',  'fan_in_bins_3-inf',
    'fan_out_bins_2-3', 'fan_out_bins_3-inf',
    'scatter-gather_bins_2-3', 'scatter-gather_bins_3-inf',
    'temp-cycle_bins_2-3',     'temp-cycle_bins_3-inf',
    'source_fan_out',  'source_fan_in',
    'dest_fan_out',    'dest_fan_in',
    'source_sum_col3_out',  'dest_sum_col3_in',
]
# Only check columns that actually exist (in case params change the bins)
signal_cols = [c for c in signal_cols if c in edge_features.columns]

legit = edge_features[edge_features['label'] == 0][signal_cols]
laund = edge_features[edge_features['label'] == 1][signal_cols]
comp  = pd.DataFrame({
    'Legit mean'      : legit.mean(),
    'Laundering mean' : laund.mean(),
    'Ratio (L/l)'     : (laund.mean() / (legit.mean() + 1e-9)).round(2),
})
print('Sanity check — laundering vs legitimate feature means (ratio > 1 = good signal):')
print(comp.round(4).to_string())

In [ ]:
# ── 4.3  Save CSV files ───────────────────────────────────────────────────
edge_features.to_csv('Data/edge_features_gfp.csv', index=False)
node_features.to_csv('Data/node_features_gfp.csv', index=False)

print('Saved:')
print(f'  Data/edge_features_gfp.csv  — {edge_features.shape}')
print(f'  Data/node_features_gfp.csv  — {node_features.shape}')
vc = edge_features['label'].value_counts()
print(f'\nClass balance: {vc[1]:,} laundering / {vc[0]:,} legitimate '
      f'({100*vc[1]/vc.sum():.4f}%)')

---
## 5. Temporal 60 / 20 / 20 split


In [ ]:
edge_df = edge_features.copy()
edge_df['Timestamp'] = pd.to_datetime(edge_df['Timestamp'])
edge_df = edge_df.sort_values('Timestamp', kind='stable').reset_index(drop=True)

n_edges = len(edge_df)
t1_idx  = int(n_edges * 0.60)
t2_idx  = int(n_edges * 0.80)

t1 = edge_df.loc[t1_idx - 1, 'Timestamp']
t2 = edge_df.loc[t2_idx - 1, 'Timestamp']

train_mask = edge_df.index < t1_idx
val_mask   = (edge_df.index >= t1_idx) & (edge_df.index < t2_idx)
test_mask  = edge_df.index >= t2_idx

print('Temporal split (60 / 20 / 20):')
print(f'  t1 (train end) : {t1}  →  {t1_idx:,} training edges')
print(f'  t2 (val end)   : {t2}  →  {t2_idx - t1_idx:,} validation edges')
print(f'  t_max          : {edge_df["Timestamp"].max()}  →  {n_edges - t2_idx:,} test edges')
print(f'\n  Laundering in train : {edge_df.loc[train_mask, "label"].sum():,} | Rate: {edge_df.loc[train_mask, "label"].mean()*100:.4f}%')
print(f'  Laundering in val   : {edge_df.loc[val_mask,   "label"].sum():,}   | Rate: {edge_df.loc[val_mask,   "label"].mean()*100:.4f}%')
print(f'  Laundering in test  : {edge_df.loc[test_mask,  "label"].sum():,}   | Rate: {edge_df.loc[test_mask,  "label"].mean()*100:.4f}%')

---
## 6. Graph construction

Same cumulative-snapshot protocol as `Data_prepration.ipynb`:
- **train_graph:** train edges only, all evaluated
- **val_graph:** train + val edges, evaluated on val portion
- **test_graph:** all edges, evaluated on test portion


In [ ]:
# ── 6.1  Node feature matrix ──────────────────────────────────────────────
all_accounts = pd.concat([
    node_features['account_id'],
    edge_df['src_account'],
    edge_df['dst_account'],
]).unique()

# Rebuild account_to_idx to include all accounts (edges may have accounts
# not in the accounts table)
account_to_idx = {acc: idx for idx, acc in enumerate(all_accounts)}
N_nodes = len(account_to_idx)
print(f'Total unique accounts (nodes): {N_nodes:,}')

# Handle duplicate account_ids
nf_deduped = (
    node_features.drop_duplicates(subset='account_id', keep='first')
    if node_features['account_id'].duplicated().sum() > 0
    else node_features
)

idx_series    = pd.Series(account_to_idx)          # account_id → integer index
node_feat_arr = (
    nf_deduped
    .set_index('account_id')
    .reindex(idx_series.index)    # align to account_to_idx order
    [NODE_FEAT_COLS]
    .fillna(0)
    .values
    .astype(np.float32)
)

X_node = torch.tensor(node_feat_arr, dtype=torch.float)
print(f'Node feature matrix : {tuple(X_node.shape)}')

In [ ]:
# ── 6.2  Graph builder function ───────────────────────────────────────────
def build_graph(edge_subset, eval_mask):
    """
    Build a PyG Data object from a subset of edges.

    Parameters
    ----------
    edge_subset : pd.DataFrame  — rows from edge_df
    eval_mask   : np.ndarray[bool] — which rows are in the evaluation set

    Returns
    -------
    torch_geometric.data.Data
    """
    src = edge_subset['src_account'].map(account_to_idx).values
    dst = edge_subset['dst_account'].map(account_to_idx).values

    edge_index = torch.tensor(np.stack([src, dst], axis=0), dtype=torch.long)
    edge_attr  = torch.tensor(
        edge_subset[EDGE_FEAT_COLS].values.astype(np.float32), dtype=torch.float
    )
    edge_time  = torch.tensor(
        edge_subset['Timestamp'].astype('int64').values // 10**9, dtype=torch.long
    )

    # Labels: -1 for context edges, 0/1 for evaluated edges
    labels          = np.full(len(edge_subset), -1, dtype=np.int64)
    labels[eval_mask] = edge_subset.loc[eval_mask, 'label'].values.astype(np.int64)

    return Data(
        x          = X_node,
        edge_index = edge_index,
        edge_attr  = edge_attr,
        edge_time  = edge_time,
        y          = torch.tensor(labels, dtype=torch.long),
        eval_mask  = torch.tensor(eval_mask, dtype=torch.bool),
        num_nodes  = N_nodes,
    )

In [ ]:
# ── 6.3  Build snapshots ──────────────────────────────────────────────────
print('Building graph snapshots ...')

with Timer('train graph'):
    train_edges = edge_df[train_mask].reset_index(drop=True)
    train_eval  = np.ones(len(train_edges), dtype=bool)
    train_graph = build_graph(train_edges, train_eval)

with Timer('val graph'):
    val_df   = edge_df[train_mask | val_mask].reset_index(drop=True)
    val_eval = np.zeros(len(val_df), dtype=bool)
    val_eval[t1_idx:] = True
    val_graph = build_graph(val_df, val_eval)

with Timer('test graph'):
    all_df    = edge_df.reset_index(drop=True)
    test_eval = np.zeros(len(all_df), dtype=bool)
    test_eval[t2_idx:] = True
    test_graph = build_graph(all_df, test_eval)

print('All snapshots built ✓')

In [ ]:
# ── 6.4  Summary ─────────────────────────────────────────────────────────
def summarise(name, g):
    n_eval  = g.eval_mask.sum().item()
    n_laund = (g.y[g.eval_mask] == 1).sum().item()
    rate    = n_laund / n_eval * 100 if n_eval > 0 else 0
    print(f'  {name:<14} | nodes={g.num_nodes:>7,} | edges={g.edge_index.shape[1]:>9,} '
          f'| eval={n_eval:>9,} | laund={n_laund:>5,} ({rate:.4f}%)')

print('\n── Graph snapshots ────────────────────────────────────────────────────')
summarise('train_graph', train_graph)
summarise('val_graph',   val_graph)
summarise('test_graph',  test_graph)
print(f'\nEdge feature dim : {train_graph.edge_attr.shape[1]}')
print(f'Node feature dim : {train_graph.x.shape[1]}')

In [ ]:
# ── 6.5  Save ─────────────────────────────────────────────────────────────
torch.save(train_graph, 'Data/train_graph_gfp.pt')
torch.save(val_graph,   'Data/val_graph_gfp.pt')
torch.save(test_graph,  'Data/test_graph_gfp.pt')

with open('Data/account_to_idx_gfp.pkl', 'wb') as f:
    pickle.dump(account_to_idx, f)

# Reload check
_g = torch.load('Data/train_graph_gfp.pt', weights_only=False)
assert _g.edge_attr.shape[1] == len(EDGE_FEAT_COLS), \
    f'Feature dim mismatch: {_g.edge_attr.shape[1]} vs {len(EDGE_FEAT_COLS)}'

print('Saved:')
print('  Data/train_graph_gfp.pt')
print('  Data/val_graph_gfp.pt')
print('  Data/test_graph_gfp.pt')
print('  Data/account_to_idx_gfp.pkl')
print(f'\nReload check → edges: {_g.edge_index.shape[1]:,}  edge_attr: {_g.edge_attr.shape}  ✓')
print()
print('═' * 60)
print(f'  ACTION: set  EDGE_DIM = {train_graph.edge_attr.shape[1]}')
print(f'          set  NODE_DIM = {train_graph.x.shape[1]}')
print(f'          in your model notebook (baseline_model_gfp.ipynb)')
print('═' * 60)

---
## Appendix — Sanity checks


In [ ]:
# Check 1: FX conversion — verify USD amounts are reasonable
print('Check 1 — FX conversion sanity:')
print(edges.groupby('Payment Currency')['Amount_USD'].describe()[['mean', 'min', 'max']].round(2))

In [ ]:
# Check 2: GFP causal correctness — first transaction for each src must have 0 history stats
first_txn_mask = edges.groupby('src_account').cumcount() == 0
history_cols   = [c for c in GFP_FEAT_COLS if 'sum' in c or 'avg' in c or 'fan' in c]
history_cols   = history_cols[:5]  # check a sample

first_vals = edge_features.loc[first_txn_mask, history_cols]
print('Check 2 — GFP history stats are 0 on first transaction per account (causal guard):')
print(first_vals.describe().loc[['max']].round(4))
print(f'\nAll zero: {(first_vals == 0).all().all()} ✓' if (first_vals == 0).all().all()
      else '\nNon-zero values detected — check GFP causal setting!')

In [ ]:
# Check 3: feature correlation heatmap (top GFP columns vs label)
corr_cols  = GFP_FEAT_COLS[:10] + ['label']
corr_check = [c for c in corr_cols if c in edge_features.columns]
print('Check 3 — Pearson correlation of GFP features with label:')
print(
    edge_features[corr_check]
    .corr()['label']
    .drop('label')
    .sort_values(ascending=False)
    .round(4)
    .to_string()
)